# The Engine Changed Under Us: What kaggle-environments 1.32.6 Did to the Economy

If you tuned an agent before **August 6-7**, some of your numbers are describing a game
that no longer exists. `kaggle-environments` **1.32.6** rebalanced town demand, and the
change is large enough to move which crops are worth planting.

I found this the hard way. I had spent two days fitting crop targets and sell thresholds
against **1.32.5**, and when I upgraded, my agent's bank fell by half while my opponents'
fell by a third. This notebook is the diff, quantified, so nobody else has to lose the
time.

Everything below is computed from the installed package — you can re-run it and check me.

*I'm a high-school student doing this competition as an economics project. If something
here is wrong, please say so in the comments.*

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable,"-m","pip","show","kaggle-environments"],
                     capture_output=True,text=True).stdout.split("Location")[0].strip())

## 1. What actually changed

Three edits to `kaggriculture.py`:

| | 1.32.5 | 1.32.6 |
|---|---|---|
| `townCenterSellInterval` default | 12 turns (2 ticks/day) | **24 turns (1 tick/day)** |
| `TOWN_CENTER_DEMAND_SCHEDULE` | `[(20,4), (10,2), (0,1)]` — 1x, then 2x from day 10, 4x from day 20 | **removed — flat 1 per tick** |
| shop unlocks | 8 distinct shops, sampled without replacement | **sampled WITH replacement**, capped by `MAX_SHOP_INSTANCES = 8` |

The first two compound. The town centre used to consume a product **twice a day**, at a
multiplier that reached **4x** in the last third of the season — up to **8 units/day**. It
now consumes **1 unit/day, flat, all season.**

In [ ]:
from kaggle_environments.envs.kaggriculture.kaggriculture import (
    SHOPS, PRODUCTS, MARKET_I0, market_price,
)
try:
    from kaggle_environments.envs.kaggriculture.kaggriculture import MAX_SHOP_INSTANCES
    NEW = True
except ImportError:
    MAX_SHOP_INSTANCES, NEW = None, False
print("running on the NEW (1.32.6+) engine" if NEW else "running on an OLD (<=1.32.5) engine")
print("MAX_SHOP_INSTANCES =", MAX_SHOP_INSTANCES)

TURNS_PER_DAY, DAYS = 24, 30
SHOP_TICKS_PER_DAY = TURNS_PER_DAY // 4          # shops consume every 4 turns

## 2. Town demand, before and after

Shops still consume every 4 turns (6 ticks/day), doubled for single-product shops. But
they are now drawn **with replacement**, so instead of "all 8 shops, one each", you get 8
draws from 8 shops — the same shop can appear several times and some never appear at all.

The expected per-product shop demand is therefore `8 draws x (1/8 chance) x 6 ticks x mult`,
which happens to equal the old all-distinct figure *in expectation* — but with far more
variance between episodes. The real cut comes from the town centre.

In [ ]:
exp_shop = {p: 0.0 for p in PRODUCTS}
for shop, prods in SHOPS.items():
    mult = 2 if len(prods) == 1 else 1
    for p in prods:
        exp_shop[p] += (MAX_SHOP_INSTANCES or 8) * (1/len(SHOPS)) * SHOP_TICKS_PER_DAY * mult

# town centre: OLD = 2 ticks/day x {1,2,4} by day; NEW = 1 tick/day x 1
old_centre_season = sum(2 * (4 if d>=20 else 2 if d>=10 else 1) for d in range(DAYS))
new_centre_season = DAYS * 1

print(f"town-centre consumption per product, whole season:")
print(f"   1.32.5 : {old_centre_season} units")
print(f"   1.32.6 : {new_centre_season} units      ({new_centre_season/old_centre_season-1:+.0%})")
print()
print(f"{'product':<12}{'shops/day':>10}{'centre/day OLD':>16}{'centre/day NEW':>16}"
      f"{'sink OLD':>10}{'sink NEW':>10}{'change':>9}")
for p in ["WHEAT","STRAWBERRY","MILK","CARROT","WOOL","EGG","TOMATO","MELON"]:
    old_c, new_c = 8, 1          # late-season town-centre rate, old vs new
    o, n = exp_shop[p] + old_c, exp_shop[p] + new_c
    print(f"{p:<12}{exp_shop[p]:>10.0f}{old_c:>16}{new_c:>16}{o:>10.0f}{n:>10.0f}{n/o-1:>8.0%}")

### The headline: melon lost 88% of its demand

Melon is the only product **no shop wants** — its sole buyer was ever the town centre. It
used to get up to 8 units/day late in the season. It now gets **one unit per day, for the
whole game.**

Everything else lost 18-35%, roughly in proportion to how much of its demand came from the
centre rather than from shops.

## 3. Why this changes what you plant

Town consumption is a *sink*: it removes inventory permanently and for free, which is what
lets a price recover after you sell into it. Supply below the sink keeps a price above
base; supply above it grinds the price down.

Shrinking every sink therefore does two things at once:

1. **Gluts arrive sooner and bite harder.** The same selling schedule that was fine on
   1.32.5 now walks further down the price curve.
2. **Products whose demand was mostly town-centre are hit hardest.** That is melon, and to
   a lesser degree wool, egg and tomato.

Here is what a fixed selling schedule earns before and after, holding your behaviour
constant — the curve is unchanged, but you reach the bad part of it sooner.

**The effect is strongly volume-dependent, and that is the actual lesson.** If you sell
comfortably under the sink, the rebalance barely touches you. The damage scales with how
far above the sink you were already selling — so it punishes exactly the high-output
strategies that were winning before.

In [ ]:
def realised(item, qty, drain_per_day, days=DAYS):
    """Sell `qty` units evenly across the season while the town drains `drain_per_day`."""
    inv, total, per_day = MARKET_I0, 0, qty/days
    carry = 0.0
    for d in range(days):
        carry += per_day
        while carry >= 1:
            total += market_price(item, int(inv)); inv += 1; carry -= 1
        inv -= drain_per_day                        # town eats
    return total/qty if qty else 0

print("avg $/unit realised, selling QTY evenly across the season   (OLD -> NEW, % change)\n")
print(f"{'product':<12}" + "".join(f"{f'Q={q}':>22}" for q in (100, 300, 600)))
for p in ["MELON","WOOL","STRAWBERRY","MILK","WHEAT"]:
    cells = []
    for q in (100, 300, 600):
        o = realised(p, q, exp_shop[p] + 4)   # old: centre averaged ~4-5/day over the season
        n = realised(p, q, exp_shop[p] + 1)
        cells.append(f"{o:>7.0f} ->{n:>6.0f} ({n/o-1:>+4.0%})")
    print(f"{p:<12}" + "".join(f"{c:>22}" for c in cells))

## 4. Practical consequences

- **Re-check your version before you trust a sweep.** `pip show kaggle-environments`. If it
  says 1.32.5 or lower, your tuning set is describing the old economy. This cost me two
  days of A/B results.
- **Melon is now a very small tranche or none at all.** One unit of demand per day, and the
  harshest glut curve in the game (`above_func="sq"`, `above_target=3.6`).
- **Anything you sell on a fixed schedule needs re-checking**, because the schedule was
  implicitly calibrated to a sink that is now 18-88% smaller.
- **Absolute price thresholds age badly.** If your agent holds stock until a product clears
  a fixed dollar figure, the lower price level means it now holds too long and dumps at the
  end. Thresholds relative to the current price survive a rebalance; absolute ones don't.
- **Shops drawn with replacement means higher variance between episodes.** Two episodes can
  now have very different demand mixes, so you need more seeds to measure the same effect.

## 5. Caveats

- The shop figures are **expectations**. With replacement, any single episode's mix can be
  far from it — that is precisely the new variance.
- The "old centre ≈ 4/day" used in section 3 is a season average of the 1/2/4 schedule; the
  old rate was 2/day early and 8/day late, so the comparison is indicative, not exact.
- I have not tried to measure how the rebalance changed the *relative* standing of
  strategies. My own agent lost about half its bank and the strong public agents lost about
  a third, but that is one data point from one agent pair and I would not generalise from it.

If this saved you some time, the thing I'd most like back is a correction if any of it is
wrong.